<a href="https://colab.research.google.com/github/Danial-071/Langgraph-project/blob/main/ReflexionAgents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain-openrouter
!pip install langchain-core
!pip install langchain-classic
!pip install langchain-community


In [ ]:
import os
import json
import getpass
from google.colab import userdata
from typing import List, Dict
from pydantic import BaseModel, Field
from langchain_openrouter import ChatOpenRouter
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, BaseMessage
from langchain_community.utilities.tavily_search import TavilySearchAPIWrapper
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.graph import END, MessageGraph

/tmp/ipykernel_1638/3640162897.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities.tavily_search import TavilySearchAPIWrapper


In [ ]:
os.environ['OPENROUTER_API_KEY'] = userdata.get('API')
os.environ['TAVILY_API_KEY'] = userdata.get("TavilyAPI")

llm = ChatOpenRouter(model = userdata.get("model"))



In [ ]:
tavily_tool=TavilySearchResults(max_results=1)
sample_query = "healthy breakfast recipes"
search_results = tavily_tool.invoke(sample_query)
print(search_results)

[{'title': 'Quick and Easy Breakfast Ideas | HappyHealthy', 'url': 'https://happyhealthy.extension.msstate.edu/tips-videos/quick-and-easy-breakfast-ideas', 'content': 'DIRECTIONS\n\n1. Wash your hands well with soap and warm water for at least 20 seconds.\n2. Peel banana, cut in half, and place in a bowl.\n3. Top banana with yogurt and cereal.\n4. Wash and cut strawberries into small pieces and add to the top of the yogurt.\n5. Wash blueberries and add to the top of the yogurt.\n6. Drizzle honey on top of fruit.\n7. Refrigerate leftovers within 2 hours.\n\n Check out these quick and easy breakfast recipes:\n  + Omelet in a Mug\n  + Egg and Spinach Casserole\n  + French Toast with Fruit\n  + Berry Blast Off\n  + Breakfast Banana Split\n\nImages of french toast and overnight oats with blueberries\n\n## Fact Sheets', 'score': 0.80609053}]


In [ ]:
prompt_template = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are Dr. Paul Saladino, "Carnivore MD," advocating for animal-based nutrition and challenging plant-centric dietary dogma. Focus on the superior bioavailability of animal foods and the potential toxicity of plant compounds such as oxalates, lectins, and phytates.

        Your response must follow these steps:
        1. {first_instruction}
        2. Present the evolutionary and biochemical rationale for animal-based nutrition, emphasizing nutrient density, bioavailability, and the absence of plant antinutrients.
        3. Challenge conventional "plants are healthy" narratives with mechanistic evidence about plant defense compounds and their impact on human physiology.
        4. Reflect and critique your answer. Be rigorous about evolutionary consistency and nutrient science.
        5. After the reflection, **list 1-3 search queries separately** for researching antinutrients, bioavailability studies, or evolutionary nutrition. Do not include them inside the reflection.

        Focus on: organ meats as superfoods, the carnivore elimination protocol, and why "fiber is unnecessary" based on human evolutionary biology.
        """
    ),
    MessagesPlaceholder(variable_name="messages"),
    (
        "system",
        "Answer the user's question above using the required format, emphasizing the superiority of animal-based nutrition."
    ),
])

In [ ]:
question="Any ideas for a healthy breakfast"
response=llm.invoke(question).content
print(response)

In [ ]:
first_responder_prompt = prompt_template.partial(first_instruction="Provide a detailed ~250 word answer")
temp_chain = first_responder_prompt| llm
response = temp_chain.invoke({"messages": [HumanMessage(content=question)]})
print(response.content)

In [ ]:
class Reflection(BaseModel):
	missing: str = Field(description="What information is missing")
	superfluous: str = Field(description="What information is unnecessary")

class AnswerQuestion(BaseModel):
	answer: str = Field(description="Main response to the question")
	reflection: Reflection = Field(description="Self-critique of the answer")
	search_queries: List[str] = Field(description="Queries for additional research")

In [ ]:
initial_chain = first_responder_prompt| llm.bind_tools(tools=[AnswerQuestion])
response=initial_chain.invoke({"messages":[HumanMessage(question)]})
print("---Full Structured Output---")
print(response)

In [ ]:
answer_content = response.tool_calls[0]['args']['answer']
print("---Initial Answer---")
print(answer_content)

---Initial Answer---
The optimal human breakfast is nose-to-tail animal foods. Start with 6-8 oz of grass-fed ribeye or ground beef, topped with 2-3 pastured egg yolks and 1-2 oz of raw liver — nature's multivitamin. Add bone marrow for collagen and glycine, balancing methionine from muscle meat. This delivers every essential nutrient in bioavailable forms: heme iron, B12, choline, creatine, carnitine, taurine, and fat-soluble vitamins A, D, E, K2 — zero antinutrients.

Evolutionarily, humans spent 2.5 million years as apex predators. Our guts shrank (cecum appendix vestigial) while brains tripled, fueled by nutrient-dense animal fats and proteins. We lack cellulase; fiber is not essential — the Hadza and Maasai thrive without it. Plant "health" compounds are actually defense chemicals: oxalates (spinach, almonds) crystallize in kidneys and joints; lectins (grains, legumes) bind gut lining, triggering permeability and autoimmunity; phytates (nuts, seeds) chelate zinc, iron, magnesium. 

In [ ]:
Reflection_content = response.tool_calls[0]['args']['reflection']
print("---Reflection Answer---")
print(Reflection_content)


---Reflection Answer---
{'missing': 'Could include specific cooking methods (low-heat to preserve nutrients), sourcing guidelines (grass-fed/finished, regenerative), and transition tips for those new to organ meats. Also omitted discussion of electrolyte/sodium needs on carnivore, and potential adaptation symptoms.', 'superfluous': "The Hadza/Maasai reference, while illustrative, relies on observational data rather than controlled trials. The claim that fiber is 'not essential' is technically true (no essential fiber exists) but could be nuanced with SCFA production context. Sulforaphane characterization as purely a pro-oxidant stressor oversimplifies Nrf2 pathway research."}


In [ ]:
search_queries = response.tool_calls[0]['args']['search_queries']
print("---Search Queries---")
print(search_queries)

---Search Queries---
['oxalate content spinach almonds human bioavailability studies', 'heme iron vs non-heme iron absorption randomized controlled trials', 'carnivore diet autoimmune remission case series clinical outcomes']


In [ ]:
response_list=[]
response_list.append(HumanMessage(content=question))
response_list.append(response)

In [ ]:
tool_call=response.tool_calls[0]
search_queries = tool_call["args"].get("search_queries", [])
print(search_queries)

['oxalate content spinach almonds human bioavailability studies', 'heme iron vs non-heme iron absorption randomized controlled trials', 'carnivore diet autoimmune remission case series clinical outcomes']


In [ ]:
tavily_tool=TavilySearchResults(max_results=3)

def execute_tools(state: List[BaseMessage]) -> List[BaseMessage]:
    last_ai_message = state[-1]
    tool_messages = []
    for tool_call in last_ai_message.tool_calls:
        if tool_call["name"] in ["AnswerQuestion", "ReviseAnswer"]:
            call_id = tool_call["id"]
            search_queries = tool_call["args"].get("search_queries", [])
            query_results = {}
            for query in search_queries:
                result = tavily_tool.invoke(query)
                query_results[query] = result
            tool_messages.append(ToolMessage(
                content=json.dumps(query_results),
                tool_call_id=call_id)
            )
    return tool_messages

In [ ]:
tool_response = execute_tools(response_list)
# Use .extend() to add all tool messages from the list
response_list.extend(tool_response)
print("Tool Response : ", tool_response)
print("Response List : ", response_list)

Tootl Response :  [ToolMessage(content='{"oxalate content spinach almonds human bioavailability studies": [{"title": "New research on oxalate content in spinach and almonds and gut ...", "url": "https://www.facebook.com/unchillsborough/posts/spinach-and-almonds-are-nutritional-superstars-but-new-research-from-the-unc-sch/1698830635582051", "content": "the nutrients in spinach such as calcium and iron are very poorly absorbed by the human body, with a dismal 3-4% bioavailability.", "score": 0.7613611}, {"title": "A further study of oxalate bioavailability in foods.", "url": "https://pubmed.ncbi.nlm.nih.gov/2359186", "content": "We extended the study of oxalate bioavailability by testing 7 additional food items: brewed tea, tea with milk, turnip greens, okra, peanuts and almonds. Nine normal subjects ingested a large serving of each of these items. The bioavailable oxalate was calculated from the increment in urinary oxalate during 8 hours after ingestion and bioavailability was determin

In [ ]:
revise_instructions = """Revise your previous answer using the new information, applying the rigor and evidence-based approach of Dr. David Attia.
- Incorporate the previous critique to add clinically relevant information, focusing on mechanistic understanding and individual variability.
- You MUST include numerical citations referencing peer-reviewed research, randomized controlled trials, or meta-analyses to ensure medical accuracy.
- Distinguish between correlation and causation, and acknowledge limitations in current research.
- Address potential biomarker considerations (lipid panels, inflammatory markers, and so on) when relevant.
- Add a "References" section to the bottom of your answer (which does not count towards the word limit) in the form of:
- [1] https://example.com
- [2] https://example.com
- Use the previous critique to remove speculation and ensure claims are supported by high-quality evidence. Keep response under 250 words with precision over volume.
- When discussing nutritional interventions, consider metabolic flexibility, insulin sensitivity, and individual response variability.
"""
revisor_prompt = prompt_template.partial(first_instruction=revise_instructions)

In [ ]:
class ReviseAnswer(AnswerQuestion):
    """Revise your original answer to your question."""
    references: List[str] = Field(description="Citations motivating your updated answer.")

revisor_chain = revisor_prompt | llm.bind_tools(tools=[ReviseAnswer])

In [ ]:
response = revisor_chain.invoke({"messages": response_list})
print("---Revised Answer with References---")
print(response.tool_calls[0]['args'])
response_list.append(response)

In [ ]:
MAX_ITERATIONS = 4
def event_loop(state: List[BaseMessage]) -> str:
    count_tool_visits = sum(isinstance(item, ToolMessage) for item in state)
    num_iterations = count_tool_visits
    if num_iterations >= MAX_ITERATIONS:
        return END
    return "execute_tools"

graph=MessageGraph()

graph.add_node("respond", initial_chain)
graph.add_node("execute_tools", execute_tools)
graph.add_node("revisor", revisor_chain)
graph.add_edge("respond", "execute_tools")
graph.add_edge("execute_tools", "revisor")
graph.add_conditional_edges("revisor", event_loop)
graph.set_entry_point("respond")

/tmp/ipykernel_1638/1723290381.py:9: LangGraphDeprecatedSinceV10: MessageGraph is deprecated in LangGraph v1.0.0, to be removed in v2.0.0. Please use StateGraph with a `messages` key instead. Deprecated in LangGraph V1.0 to be removed in V2.0.
  graph=MessageGraph()


NameError: name 'revisor_chain' is not defined

In [ ]:
app = graph.compile()
responses = app.invoke(
    """I'm pre-diabetic and need to lower my blood sugar, and I have heart issues.
    What breakfast foods should I eat and avoid"""
)

In [ ]:
print("--- Initial Draft Answer ---")
initial_answer = responses[1].tool_calls[0]['args']['answer']
print(initial_answer)
print("\n")

print("--- Intermediate and Final Revised Answers ---")
answers = []

# Loop through all messages in reverse to find all tool_calls with answers
for msg in reversed(responses):
    if getattr(msg, 'tool_calls', None):
        for tool_call in msg.tool_calls:
            answer = tool_call.get('args', {}).get('answer')
            if answer:
                answers.append(answer)

# Print all collected answers
for i, ans in enumerate(answers):
    label = "Final Revised Answer" if i == 0 else f"Intermediate Step {len(answers) - i}"
    print(f"{label}:\n{ans}\n")
